# 📊 Employee Productivity & Quality Analysis
## Complete Analysis Notebook
### Vinod M | Data Analyst | Q1 2026

## 1. Setup & Data Loading

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Settings
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

DATA_PATH = '../data/raw/employee_productivity_dataset.csv'
df = pd.read_csv(DATA_PATH)
df['Date'] = pd.to_datetime(df['Date'])
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '../data/raw/employee_productivity_dataset.csv'

## 2. Data Quality Check

In [ ]:
print('=== MISSING VALUES ===')
print(df.isnull().sum())
print(f'\n=== DUPLICATES === {df.duplicated().sum()}')
print('\n=== DATA TYPES ===')
print(df.dtypes)
print('\n=== STATISTICAL SUMMARY ===')
df.describe().round(2)

## 3. KPI Creation

In [ ]:
# KPI 1: Productivity %
df['Productivity_Pct'] = (df['Tasks_Completed'] / df['Target_Tasks'] * 100).round(2)

# KPI 2: Efficiency (Tasks per Hour)
df['Efficiency'] = (df['Tasks_Completed'] / df['Hours_Worked']).round(2)

# Time columns
df['Month']      = df['Date'].dt.month
df['Month_Name'] = df['Date'].dt.strftime('%b %Y')

# Performance Category
def perf_cat(p):
    if p >= 110: return 'Excellent'
    elif p >= 100: return 'Good'
    elif p >= 90: return 'Average'
    elif p >= 80: return 'Below Average'
    return 'Poor'
df['Performance_Category'] = df['Productivity_Pct'].apply(perf_cat)

print('Company KPI Summary:')
print(f'  Avg Productivity : {df["Productivity_Pct"].mean():.2f}%  (Target: 100%)')
print(f'  Avg Quality      : {df["Quality_Score"].mean():.2f}%   (Target: 95%)')
print(f'  Avg Efficiency   : {df["Efficiency"].mean():.2f} tasks/hr')
print(f'  Total Tasks      : {df["Tasks_Completed"].sum():,}')
print(f'  Total Hours      : {df["Hours_Worked"].sum():,}')

## 4. Department Analysis

In [ ]:
dept = df.groupby('Department').agg(
    Employees       =('Employee_ID','nunique'),
    Avg_Productivity=('Productivity_Pct','mean'),
    Avg_Quality     =('Quality_Score','mean'),
    Avg_Efficiency  =('Efficiency','mean'),
    Total_Tasks     =('Tasks_Completed','sum')
).round(2).sort_values('Avg_Productivity', ascending=False)
print(dept)

COLORS = {'AI':'#1565C0','Data':'#7C3AED','QA':'#0D9488','Operations':'#EA580C'}
DEPTS  = ['AI','Data','QA','Operations']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col, title, target in [
    (axes[0], 'Avg_Productivity', 'Productivity %', 100),
    (axes[1], 'Avg_Quality',      'Quality %',      95),
    (axes[2], 'Avg_Efficiency',   'Efficiency',     None),
]:
    vals = dept.reindex(DEPTS)[col]
    bars = ax.bar(vals.index, vals.values, color=[COLORS[d] for d in DEPTS], width=0.5, edgecolor='white')
    for b in bars:
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.2, f'{b.get_height():.2f}', ha='center', fontsize=9, fontweight='bold')
    if target: ax.axhline(target, color='red', ls='--', lw=1.5, label=f'Target {target}')
    ax.set_title(f'{title} by Department', fontweight='bold'); ax.legend(fontsize=8)
fig.suptitle('Department Performance Comparison', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 5. Employee Analysis — Top & Bottom Performers

In [ ]:
emp = df.groupby(['Employee_Name','Department']).agg(
    Avg_Productivity=('Productivity_Pct','mean'),
    Avg_Quality     =('Quality_Score','mean'),
    Avg_Efficiency  =('Efficiency','mean'),
).round(2).reset_index()

top10 = emp.sort_values('Avg_Productivity', ascending=False).head(10)
bot10 = emp.sort_values('Avg_Productivity', ascending=True).head(10)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
ax1.barh(top10['Employee_Name'][::-1], top10['Avg_Productivity'][::-1], color='#1565C0', edgecolor='white')
ax1.axvline(100, color='red', ls='--', lw=1.5, label='Target')
ax1.set_title('Top 10 Employees — Productivity %', fontweight='bold'); ax1.legend()
for i, v in enumerate(top10['Avg_Productivity'][::-1]):
    ax1.text(v+0.3, i, f'{v:.2f}%', va='center', fontsize=9)

ax2.barh(bot10['Employee_Name'], bot10['Avg_Productivity'], color='#DC2626', edgecolor='white')
ax2.axvline(100, color='green', ls='--', lw=1.5, label='Target')
ax2.set_title('Bottom 10 Employees — Needs Training', fontweight='bold'); ax2.legend()
for i, v in enumerate(bot10['Avg_Productivity']):
    ax2.text(v+0.3, i, f'{v:.2f}%', va='center', fontsize=9)

plt.tight_layout(); plt.show()
print(f'Top Performer : {top10.iloc[0]["Employee_Name"]} — {top10.iloc[0]["Avg_Productivity"]:.2f}%')
print(f'Low Performer : {bot10.iloc[0]["Employee_Name"]} — {bot10.iloc[0]["Avg_Productivity"]:.2f}%')

## 6. Monthly Trend Analysis

In [ ]:
monthly = df.groupby(['Month','Month_Name']).agg(
    Avg_Productivity=('Productivity_Pct','mean'),
    Avg_Quality     =('Quality_Score','mean'),
    Avg_Efficiency  =('Efficiency','mean'),
    Total_Tasks     =('Tasks_Completed','sum')
).round(2).reset_index().sort_values('Month')
print(monthly[['Month_Name','Avg_Productivity','Avg_Quality','Avg_Efficiency','Total_Tasks']].to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
months = monthly['Month_Name'].tolist()
for ax, col, title, target, color in [
    (axes[0], 'Avg_Productivity', 'Productivity %', 100, '#1565C0'),
    (axes[1], 'Avg_Quality',      'Quality %',      95,  '#7C3AED'),
    (axes[2], 'Avg_Efficiency',   'Efficiency',     None,'#EA580C'),
]:
    ax.plot(months, monthly[col], marker='o', color=color, lw=2.5, ms=9)
    for x, y in zip(months, monthly[col]):
        ax.annotate(f'{y:.2f}', (x, y), textcoords='offset points', xytext=(0,10), ha='center', fontsize=10, fontweight='bold')
    if target: ax.axhline(target, color='red', ls='--', lw=1.5, label=f'Target {target}')
    ax.set_title(f'Monthly {title} Trend', fontweight='bold'); ax.tick_params(axis='x', rotation=10)
    if target: ax.legend(fontsize=9)
plt.suptitle('Monthly Performance Trends — Q1 2026', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 7. Correlation Analysis

In [ ]:
corr = df[['Tasks_Completed','Quality_Score','Hours_Worked','Productivity_Pct','Efficiency']].corr().round(3)
print('Correlation Matrix:')
print(corr)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            linewidths=0.5, linecolor='white', ax=ax)
ax.set_title('Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

r = df['Hours_Worked'].corr(df['Tasks_Completed'])
print(f'\nHours vs Tasks Correlation: {r:.3f}')
print('Interpretation:', 'Weak' if abs(r)<0.3 else 'Moderate' if abs(r)<0.7 else 'Strong', 'relationship')

## 8. Key Insights & Recommendations

In [ ]:
print('='*60)
print('KEY INSIGHTS — Q1 2026 EMPLOYEE PRODUCTIVITY ANALYSIS')
print('='*60)
print(f'''
1. OVERALL PRODUCTIVITY: {df["Productivity_Pct"].mean():.2f}% (Target: 100%)
   → Company is near target but not exceeding it consistently.

2. BEST DEPARTMENT: AI ({df[df["Department"]=="AI"]["Productivity_Pct"].mean():.2f}%)
   → Leads in productivity, quality, and efficiency.

3. NEEDS ATTENTION: Operations ({df[df["Department"]=="Operations"]["Productivity_Pct"].mean():.2f}%)
   → Below target in productivity AND quality (91.01%).

4. TOP EMPLOYEE: Employee_07 (118.67%)
   → Consistent high performer. Use as mentor.

5. TREND: Jan(95%) → Feb(99%) → Mar(102%)
   → Positive upward trend. March crossed 100% target!

RECOMMENDATIONS:
  1. Raise targets for AI & Data to 105%
  2. Training program for Operations department
  3. Coaching for bottom 5 employees
  4. Replicate March 2026 conditions in Q2
  5. Peer-mentoring: Employee_07, 12, 03 as coaches
''')